# 07 · Evaluate

Stage 8 (see `ARCHITECTURE.md`): score the whole flow on a fixed test set, so we can tell whether a change really helps instead of judging by eye.

- **Test set:** `eval/questions.json`, 8 questions with answers checked by hand against the filing. Each has an `evidence` sentence that contains the answer, and the words the answer must contain.
- **Flow:** `src/graph.py` (LangGraph): retrieve → rerank → answer → verify, and **one retry** if the checks find a problem.
- **Compared:** the old prompt (`v1`) vs. the quote-first prompt (`v2`), and fixed vs. semantic chunks.

Three scores per question:

| Score | Question it answers |
|---|---|
| `retrieval_hit` | Was the chunk with the answer among the 5 given to the model? (retrieval + rerank) |
| `correct` | Does the answer contain the right number, or say "Not found" when it should? (the LLM) |
| `passed_checks` | Did the code checks find nothing wrong? |

⚠️ Memory and time: like `06`, ~4–4.5 GB in total with Ollama; close the browser first. 8 questions × 3 setups, with retries, take roughly **8–15 minutes**.

Writes `reports/eval_results.csv`.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
import graph
import pipeline

QUESTIONS = json.loads(Path("../eval/questions.json").read_text(encoding="utf-8"))
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(exist_ok=True)

# (strategy, prompt) setups to compare. Remove some to finish faster.
CONFIGS = [("fixed", "v1"), ("fixed", "v2"), ("semantic", "v2")]

print(len(QUESTIONS), "questions ·", len(CONFIGS), "setups · model:", pipeline.LLM_MODEL)

8 questions · 3 setups · model: phi4-mini


## Scoring

Numbers are compared with commas removed, so "1,748,835" and "1748835" both count. The evidence check ignores spaces and capital letters, the same way the quote check does.

In [2]:
def score(q, state):
    result = state["result"]
    if q.get("expect_not_found"):
        hit = None  # nothing to retrieve
        correct = result["not_found"]
    else:
        evidence = pipeline.normalize(q["evidence"])
        hit = any(evidence in pipeline.normalize(doc.page_content) for _, doc in state["chunks"])
        text = result["answer"].replace(",", "")
        correct = not result["not_found"] and any(m.replace(",", "") in text for m in q["must_contain_any"])
    return {
        "retrieval_hit": hit,
        "correct": correct,
        "passed_checks": result["passed"],
        "attempts": state["attempts"],
        "seconds": state["seconds"],
        "answer": result["answer"],
        "sources": ", ".join(result["sources"]),
        "problems": "; ".join(result["problems"]),
    }

## Run

Prints one line per question as it goes: ✅ correct, ❌ wrong.

In [3]:
rows = []
for strategy, prompt in CONFIGS:
    print(f"\n=== {strategy} · {prompt} ===")
    for q in QUESTIONS:
        state = graph.ask(q["question"], strategy=strategy, prompt=prompt)
        row = {"setup": f"{strategy} · {prompt}", "id": q["id"], **score(q, state)}
        rows.append(row)
        print(f"{'✅' if row['correct'] else '❌'} {q['id']:<22} hit={row['retrieval_hit']}  "
              f"tries={row['attempts']}  {row['seconds']:>5}s  {row['answer'][:90]}")

results = pd.DataFrame(rows)
results.to_csv(REPORTS_DIR / "eval_results.csv", index=False)
print("\nSaved:", REPORTS_DIR / "eval_results.csv")


=== fixed · v1 ===


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ nim-quarter            hit=True  tries=2   24.9s  The net interest margin for this quarter, which refers to the three months ended June 30, 
✅ non-accrual            hit=True  tries=2   23.7s  As of June 30, 2026, the Company had three loans held for investment on non-accrual status
✅ dividend-q2            hit=True  tries=2   17.1s  The dividend per share declared for the quarter was $0.15 per common share. This informati
✅ loans-carrying-value   hit=True  tries=2   19.1s  The total carrying value of loans held for investment at June 30, 2026, was $1,748,835,000
✅ cecl-reserve           hit=True  tries=1   13.4s  The current expected credit loss (CECL) reserve as of June 30, 2026, is $139.1 million. Th
✅ loan-count             hit=True  tries=1   13.9s  As of June 30, 2026, the portfolio included 38 loans held for investment. [fixed-0174]
✅ borrowings-quarter     hit=True  tries=1   14.5s  The weighted average borrowings for the three months ended June 30, 2026, were $1.3 billio
✅ a

## Summary

One row per setup. `false_confidence` counts answers that **passed the checks but were wrong**: the dangerous kind, because nothing warns the user.

In [4]:
summary = results.groupby("setup", sort=False).agg(
    correct=("correct", "mean"),
    retrieval_hit=("retrieval_hit", lambda s: s.dropna().astype(bool).mean()),
    passed_checks=("passed_checks", "mean"),
    false_confidence=("correct", lambda s: int((~s & results.loc[s.index, "passed_checks"]).sum())),
    retried=("attempts", lambda s: int((s > 1).sum())),
    avg_seconds=("seconds", "mean"),
)
for col in ["correct", "retrieval_hit", "passed_checks"]:
    summary[col] = summary[col].map("{:.0%}".format)
summary["avg_seconds"] = summary["avg_seconds"].round(1)
print(summary.to_string())

              correct retrieval_hit passed_checks  false_confidence  retried  avg_seconds
setup                                                                                    
fixed · v1       100%          100%           88%                 0        4         17.1
fixed · v2        88%          100%           62%                 1        5         21.6
semantic · v2     75%           86%           25%                 0        6         31.2


## Where it went wrong

Every wrong answer, with what retrieval found and what the checks said. Read these together:

- `retrieval_hit` **False**: the answer never reached the model, so fix retrieval (chunking, reranking, more chunks).
- `retrieval_hit` **True**: the model had the answer and missed it, so fix the prompt or try a different model.

In [5]:
for row in results[~results["correct"]].to_dict("records"):
    print(f"{row['setup']} · {row['id']} · retrieval_hit={row['retrieval_hit']} · tries={row['attempts']}")
    print(f"  answer:   {row['answer']}")
    print(f"  sources:  {row['sources']}")
    print(f"  problems: {row['problems'] or 'none'}\n")

fixed · v2 · non-accrual · retrieval_hit=True · tries=2
  answer:   As of June 30, 2026, there were three loans held for investment on non-accrual status. As of December 31, 2025, there were four loans held for investment on non-accrual status.
  sources:  fixed-0063
  problems: none

semantic · v2 · non-accrual · retrieval_hit=True · tries=2
  answer:   As of June 30, 2026, three loans held for investment were on non-accrual status. As of December 31, 2025, four loans held for investment were on non-accrual status.
  sources:  semantic-0061
  problems: quote not found word for word in the cited excerpt

semantic · v2 · cecl-reserve · retrieval_hit=False · tries=2
  answer:   The current expected credit loss (CECL) reserve as of June 30, 2026, is $137,810 thousand.
  sources:  semantic-0296
  problems: quote not found word for word in the cited excerpt

